In [0]:
CATALOG = "energy_puls"
tables = ["bronze.dpe_raw", "bronze.eco2mix_raw", "silver.dpe_clean",
          "silver.meteo_clean", "silver.communes", "gold.renovation_priority"]

import pandas as pd
rows = []
for t in tables:
    d = spark.sql(f"DESCRIBE DETAIL {CATALOG}.{t}").collect()[0]
    rows.append({
        "table": t,
        "num_files": d["numFiles"],
        "size_mb": round(d["sizeInBytes"] / 1024**2, 1),
        "avg_file_mb": round(d["sizeInBytes"] / 1024**2 / max(d["numFiles"],1), 2),
    })
display(pd.DataFrame(rows))

In [0]:
import time

def timed(sql, label):
    t0 = time.time()
    n = spark.sql(sql).count()
    print(f"{label:12} {time.time()-t0:6.2f}s  ({n} rows)")

Q = f"""SELECT * FROM {CATALOG}.silver.dpe_clean
        WHERE code_commune IN ('11001','75002','44010')"""

timed(Q, "BEFORE")

In [0]:
spark.sql(f"OPTIMIZE {CATALOG}.silver.meteo_clean")

spark.sql(f"OPTIMIZE {CATALOG}.silver.dpe_clean ZORDER BY (code_commune)")
print("done")

In [0]:
timed(Q, "AFTER")

for t in ["silver.meteo_clean", "silver.dpe_clean"]:
    d = spark.sql(f"DESCRIBE DETAIL {CATALOG}.{t}").collect()[0]
    print(t, "| files:", d["numFiles"])

In [0]:
codes = [r[0] for r in spark.sql(f"""
    SELECT code_commune FROM {CATALOG}.silver.dpe_clean
    GROUP BY 1 ORDER BY COUNT(*) DESC LIMIT 3""").collect()]
print("codes:", codes)

lst = "','".join(codes)
Q2 = f"""SELECT * FROM {CATALOG}.silver.dpe_clean
         WHERE code_commune IN ('{lst}')"""
timed(Q2, "AFTER-real")

In [0]:
spark.sql(f"""CREATE OR REPLACE TABLE {CATALOG}.silver.dpe_noopt
              AS SELECT * FROM {CATALOG}.silver.dpe_clean""")

lst = "','".join(codes)
for tbl, label in [("dpe_noopt", "NO ZORDER"), ("dpe_clean", "ZORDERED ")]:
    timed(f"""SELECT * FROM {CATALOG}.silver.{tbl}
              WHERE code_commune IN ('{lst}')""", label)

for tbl in ["dpe_noopt", "dpe_clean"]:
    d = spark.sql(f"DESCRIBE DETAIL {CATALOG}.silver.{tbl}").collect()[0]
    print(tbl, "| files:", d["numFiles"])

SKEW

In [0]:
display(spark.sql(f"""
  SELECT MIN(n) AS min_rows, MAX(n) AS max_rows,
         ROUND(AVG(n),0) AS avg_rows,
         ROUND(MAX(n)/AVG(n),1) AS skew_ratio
  FROM (SELECT code_commune, COUNT(*) n
        FROM {CATALOG}.silver.dpe_clean GROUP BY 1)
"""))